# AutoML Engine Demonstration

This notebook demonstrates how to use the reusable `automl_engine`
package to train, tune, evaluate, save, reload, and use a tabular
classification model.

The machine-learning implementation is located in:

`src/automl_engine/`

This notebook acts only as a client of that package.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.datasets import load_breast_cancer

from automl_engine import (
    predict_from_run,
    run_automl,
)


# VS Code may start the notebook from either:
# 1. The project root
# 2. The notebooks directory
#
# This logic finds the project root in both cases.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# Generated datasets, predictions, and trained models
# will be stored in this ignored directory.
DEMO_DIRECTORY = PROJECT_ROOT / "demo_output"

DEMO_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


print("Python executable:")
print(sys.executable)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nDemo output directory:")
print(DEMO_DIRECTORY)

## Create a reproducible demonstration dataset

The demonstration uses scikit-learn's breast-cancer classification
dataset.

The target contains two classes:

- `malignant`
- `benign`

A categorical feature named `radius_band` is also created so the
AutoML engine demonstrates both numerical and categorical
preprocessing.

In [1]:
# Load a built-in scikit-learn dataset as pandas objects.
dataset = load_breast_cancer(
    as_frame=True
)

# dataset.frame contains the feature columns and target column.
dataframe = dataset.frame.copy()


# Replace encoded target values:
#
# 0 -> malignant
# 1 -> benign
#
# This demonstrates the engine's target-label encoding support.
target_mapping = {
    class_index: class_name
    for class_index, class_name
    in enumerate(dataset.target_names)
}

dataframe["target"] = (
    dataframe["target"]
    .map(target_mapping)
)


# Create one categorical feature from the numerical radius feature.
#
# qcut divides the observations into three similarly sized groups.
dataframe["radius_band"] = pd.qcut(
    dataframe["mean radius"],
    q=3,
    labels=[
        "small",
        "medium",
        "large",
    ],
).astype(str)


# The engine currently accepts CSV files, so save the generated
# DataFrame before starting the training workflow.
training_csv_path = (
    DEMO_DIRECTORY
    / "breast_cancer_demo.csv"
)

dataframe.to_csv(
    training_csv_path,
    index=False,
)


print("Dataset shape:", dataframe.shape)
print("Training CSV:", training_csv_path)

dataframe.head()

NameError: name 'load_breast_cancer' is not defined

In [ ]:
# Check how many rows belong to each target class.
target_distribution = (
    dataframe["target"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Rows")
)

target_distribution

## Run the AutoML workflow

The engine will now:

1. Load and validate the CSV.
2. Inspect and clean the dataset.
3. Encode the target labels.
4. Detect numerical and categorical features.
5. Create training and test sets.
6. Build preprocessing pipelines.
7. Compare candidate models using cross-validation.
8. Tune the strongest model.
9. Evaluate the final model on the untouched test set.
10. Save the fitted model and reports.

In [ ]:
result = run_automl(
    csv_path=training_csv_path,
    target_column="target",

    # We know this dataset is classification.
    # The default "auto" mode would also detect it correctly.
    task_type="classification",

    # Fast mode tunes one top-performing model using
    # a small randomized-search budget.
    tuning_mode="fast",

    # Limiting the model registry keeps this public
    # demonstration reasonably quick.
    include_models=[
        "Logistic Regression",
        "Random Forest",
    ],

    # Every run receives its own timestamped directory.
    output_directory=(
        DEMO_DIRECTORY
        / "automl_runs"
    ),

    # Three folds keep the notebook fast while still
    # demonstrating cross-validation.
    maximum_cv_folds=3,

    # Avoid nested parallel processing in the demo.
    n_jobs=1,
)

In [ ]:
run_summary = pd.Series(
    {
        "Task Type": result.task_type,
        "Final Model": result.final_model_name,
        "Selection Source": result.selection_source,
        "Run Directory": str(
            result.run_directory
        ),
        "Number of Features": len(
            result.input_schema[
                "feature_columns"
            ]
        ),
        "Numerical Features": len(
            result.input_schema[
                "numerical_columns"
            ]
        ),
        "Categorical Features": len(
            result.input_schema[
                "categorical_columns"
            ]
        ),
    },
    name="Run Summary",
)

run_summary

In [ ]:
test_metrics = pd.Series(
    result.metrics,
    name="Final Test Metrics",
)

test_metrics

In [ ]:
result.baseline_leaderboard

In [ ]:
if result.tuned_leaderboard.empty:
    print(
        "No model produced a successful "
        "tuning result."
    )

else:
    display(
        result.tuned_leaderboard
    )

In [ ]:
confusion_matrix = (
    result.evaluation_result
    .confusion_matrix
)

if confusion_matrix is not None:
    display(confusion_matrix)

In [ ]:
classification_report = (
    result.evaluation_result
    .classification_report
)

if classification_report is not None:
    display(
        classification_report.round(4)
    )

In [ ]:
prediction_preview = (
    result.evaluation_result
    .predictions
    .head(10)
)

prediction_preview

## Predict new unseen data

The fitted pipeline and target encoder were saved as artifacts.

The next cell loads those saved artifacts and makes predictions on new
rows.

An extra `sample_id` column is included. The prediction utility
preserves it in the output but does not send it into the model.

In [ ]:
# Start with five feature rows from the original dataset.
new_data = (
    dataframe
    .drop(columns=["target"])
    .sample(
        n=5,
        random_state=42,
    )
    .reset_index(drop=True)
)


# Add an extra identifier column.
#
# This was not used during training, but it should remain
# available in the prediction output.
new_data.insert(
    loc=0,
    column="sample_id",
    value=[
        1001,
        1002,
        1003,
        1004,
        1005,
    ],
)

new_data.head()

In [ ]:
new_predictions = predict_from_run(
    run_directory=(
        result.run_directory
    ),
    dataframe=new_data,
)

new_predictions

In [ ]:
probability_columns = [
    column
    for column in new_predictions.columns
    if column.startswith(
        "Probability_"
    )
]

display_columns = [
    "sample_id",
    "Prediction",
    *probability_columns,
]

new_predictions[
    display_columns
]

In [ ]:
artifact_table = pd.DataFrame(
    {
        "Artifact": list(
            result.artifacts.keys()
        ),
        "Path": [
            str(path)
            for path
            in result.artifacts.values()
        ],
    }
)

artifact_table

## Result

The demonstration completed the full workflow:

- Loaded and prepared tabular data
- Detected numerical and categorical features
- Compared classification models
- Tuned a selected model
- Evaluated the final model
- Saved all run artifacts
- Reloaded the fitted pipeline
- Generated decoded predictions and class probabilities

The notebook contains no machine-learning implementation details.
All reusable logic is maintained inside `src/automl_engine/`.